# 04. 도구와 구조화된 출력

LangChain v1에서 `@tool` 데코레이터로 커스텀 도구를 만들고, `with_structured_output()`으로 구조화된 응답을 받는 방법을 학습합니다.

## 학습 목표

- `@tool` 데코레이터로 도구를 만들고 스키마를 확인합니다
- Pydantic 모델을 사용하여 복잡한 입력 스키마를 정의합니다
- `create_agent()`에 도구를 연결하여 에이전트를 구성합니다
- `ToolRuntime`을 통해 도구에서 런타임 컨텍스트에 접근합니다
- `with_structured_output()`으로 구조화된 출력을 설정합니다
- `ToolStrategy`와 `ProviderStrategy`의 차이를 이해합니다

## 4.1 환경 설정

API 키를 로드하고 OpenAI 모델을 초기화합니다.

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

# OpenAI를 통한 모델 초기화
model = ChatOpenAI(
    model="gpt-5.4",
)

print("모델 초기화 완료:", model.model_name)

모델 초기화 완료: gpt-5.4


In [4]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — 


## 4.2 @tool 데코레이터 기본

함수에 `@tool`을 붙이면 에이전트가 사용할 수 있는 도구가 됩니다.  
LangChain은 함수의 이름, docstring, 타입 힌트를 자동으로 파싱하여 도구 스키마를 생성합니다.

```python
from langchain.tools import tool

@tool
def my_tool(param: str) -> str:
    """Tool description for the LLM."""
    return result
```

In [5]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다."""
    weather_data = {
        "Seoul": "맑음, 15\u00b0C",
        "Tokyo": "흐림, 12\u00b0C",
        "New York": "비, 8\u00b0C",
    }
    return weather_data.get(city, f"날씨 데이터를 사용할 수 없습니다: {city}")

# 도구의 스키마 확인
print("도구 이름:", get_weather.name)
print("도구 설명:", get_weather.description)
print("입력 스키마:", get_weather.args_schema.model_json_schema())

도구 이름: get_weather
도구 설명: 도시의 현재 날씨를 조회합니다.
입력 스키마: {'description': '도시의 현재 날씨를 조회합니다.', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather', 'type': 'object'}


## 4.3 Pydantic 복잡한 스키마

더 복잡한 입력 구조가 필요한 경우, Pydantic `BaseModel`을 사용하여 스키마를 정의합니다.  
`@tool(args_schema=MySchema)` 형태로 전달하면, LLM이 정확한 파라미터 구조를 이해할 수 있습니다.

- `Field(description=...)`: 각 필드에 대한 설명을 LLM에 전달
- `Field(default=...)`: 기본값 설정

In [6]:
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    """데이터베이스 쿼리용 검색 파라미터입니다."""
    query: str = Field(description="검색 쿼리 문자열")
    max_results: int = Field(default=5, description="반환할 최대 결과 수")
    category: str = Field(default="all", description="검색 카테고리: all, tech, science, news")

@tool(args_schema=SearchQuery)
def search_database(query: str, max_results: int = 5, category: str = "all") -> str:
    """고급 필터링 옵션으로 데이터베이스를 검색합니다."""
    return f"'{category}' 카테고리에서 '{query}'에 대한 {max_results}개의 결과를 찾았습니다"

print("복합 스키마:", search_database.args_schema.model_json_schema())

복합 스키마: {'description': '데이터베이스 쿼리용 검색 파라미터입니다.', 'properties': {'query': {'description': '검색 쿼리 문자열', 'title': 'Query', 'type': 'string'}, 'max_results': {'default': 5, 'description': '반환할 최대 결과 수', 'title': 'Max Results', 'type': 'integer'}, 'category': {'default': 'all', 'description': '검색 카테고리: all, tech, science, news', 'title': 'Category', 'type': 'string'}}, 'required': ['query'], 'title': 'SearchQuery', 'type': 'object'}


## 4.4 도구를 에이전트에 연결

`create_agent()`에 도구 리스트를 전달하면, 에이전트가 상황에 맞는 도구를 자동으로 선택하여 실행합니다.

```python
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[tool1, tool2],
    system_prompt="...",
)
```

> **참고:** LangChain v1에서는 `create_react_agent`가 제거되었습니다. 반드시 `create_agent`를 사용하세요.

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_weather, search_database],
    system_prompt="당신은 날씨와 검색 도구를 사용할 수 있는 어시스턴트입니다.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "서울(Seoul)의 날씨가 어떤가요?"}]},
    #{"messages": [{"role": "user", "content": "최근 10일동안 실행중인 VS_Code 툴에 접속한 유저의 숫자는?"}]},
    config=lf_config,
)
print("에이전트 응답:", result["messages"][-1].content)

에이전트 응답: 서울 현재 날씨는 **맑음, 15°C**입니다.


## 4.5 ToolRuntime

`ToolRuntime`을 사용하면 도구 함수 내에서 현재 대화 상태(state)에 접근할 수 있습니다.  
이를 통해 메시지 이력, 설정값 등 런타임 컨텍스트를 활용하는 도구를 만들 수 있습니다.

```python
@tool
def my_tool(runtime: ToolRuntime) -> str:
    messages = runtime.state["messages"]
    # ...
```

In [8]:
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(runtime: ToolRuntime) -> str:
    """현재 대화에 대한 정보를 가져옵니다."""
    messages = runtime.state["messages"]
    return f"현재 대화에 {len(messages)}개의 메시지가 있습니다."

agent_with_runtime = create_agent(
    model=model,
    tools=[get_user_info],
    system_prompt="get_user_info 도구를 사용하여 대화 정보를 확인할 수 있습니다.",
)

result = agent_with_runtime.invoke(
    {"messages": [{"role": "user", "content": "우리 대화에 메시지가 몇 개 있나요?"}]},
    config=lf_config,
)
print("응답:", result["messages"][-1].content)

응답: 현재 대화에는 2개의 메시지가 있습니다.


## 4.6 구조화된 출력

`with_structured_output()`을 사용하면 모델의 응답을 Pydantic 모델이나 dataclass 형태로 직접 받을 수 있습니다.  
이 방법은 에이전트 없이 모델에서 직접 사용합니다.

```python
structured_model = model.with_structured_output(MySchema)
result = structured_model.invoke("...")
# result는 MySchema 인스턴스
```

In [9]:
# 방법 1: with_structured_output() -- 모델 직접 사용
from pydantic import BaseModel

class MovieReview(BaseModel):
    """구조화된 영화 리뷰."""
    title: str
    rating: float
    summary: str
    recommended: bool

structured_model = model.with_structured_output(MovieReview)

review = structured_model.invoke("크리스토퍼 놀란 감독의 영화 '인셉션'을 리뷰해주세요.", config=lf_config)
print(f"제목: {review.title}")
print(f"평점: {review.rating}")
print(f"요약: {review.summary}")
print(f"추천: {'예' if review.recommended else '아니오'}")

제목: 인셉션
평점: 9.2
요약: 크리스토퍼 놀란의 '인셉션'은 꿈속의 꿈이라는 다층적 구조를 통해 현실과 무의식의 경계를 치밀하게 탐구하는 SF 스릴러입니다. 레오나르도 디카프리오를 중심으로 한 배우들의 몰입감 있는 연기, 한스 짐머의 압도적인 음악, 그리고 놀란 특유의 정교한 연출이 결합되어 강한 긴장감과 지적 재미를 동시에 제공합니다. 복잡한 서사 때문에 처음 볼 때 다소 어렵게 느껴질 수 있지만, 그만큼 다시 볼 가치가 큰 작품입니다.
추천: 예


## 4.7 ToolStrategy vs ProviderStrategy

에이전트에서 구조화된 출력을 사용하는 두 가지 전략이 있습니다:

| 전략 | 설명 | 장점 |
|------|------|------|
| `ToolStrategy` | 도구 호출 메커니즘을 활용하여 구조화된 출력 생성 | 모든 모델에서 동작, 안정적 |
| `ProviderStrategy` | 프로바이더의 네이티브 구조화 출력 기능 사용 | 더 빠르고 정확 (지원 모델 한정) |

`response_format` 파라미터에 전략을 지정하여 에이전트의 최종 응답을 구조화할 수 있습니다.

In [10]:
from langchain.agents.structured_output import ToolStrategy
from dataclasses import dataclass

@dataclass
class CalculationResult:
    """계산 결과."""
    expression: str
    result: float
    explanation: str

@tool
def calculate(expression: str) -> str:
    """수학 표현식을 계산합니다."""
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"오류: {e}"

agent_structured = create_agent(
    model=model,
    tools=[calculate],
    system_prompt="당신은 수학 어시스턴트입니다. 항상 calculate 도구를 사용하세요.",
    response_format=ToolStrategy(CalculationResult),
)

result = agent_structured.invoke(
    {"messages": [{"role": "user", "content": "2의 10제곱은 얼마인가요?"}]},
    config=lf_config,
)
print("구조화된 응답:", result.get("structured_response"))

구조화된 응답: CalculationResult(expression='2^10', result=1024.0, explanation='2를 10번 곱하면 1024입니다.')


## 4.8 요약

이 노트북에서 학습한 핵심 내용을 정리합니다.

| 항목 | 설명 |
|------|------|
| `@tool` 데코레이터 | 함수를 에이전트용 도구로 변환 |
| `args_schema` | Pydantic 모델로 복잡한 입력 스키마 정의 |
| `create_agent()` | 모델과 도구를 연결하여 에이전트 생성 |
| `ToolRuntime` | 도구 내에서 런타임 상태(대화 이력 등) 접근 |
| `with_structured_output()` | 모델 응답을 Pydantic/dataclass로 구조화 |
| `ToolStrategy` | 도구 호출 방식의 구조화된 에이전트 출력 |
| `ProviderStrategy` | 프로바이더 네이티브 구조화 출력 |

### 다음 단계
→ **[05_memory_and_streaming.ipynb](./05_memory_and_streaming.ipynb)**: 메모리와 스트리밍을 배웁니다.
